## Triple

In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time
from tqdm import tqdm
from getpass import getpass

client = OpenAI(api_key=" ",
                base_url="https://")

def get_triplets_from_ai(chunk_text):
    """Call API to extract triplets in English"""
    system_prompt = """
    You are a board-certified endocrinologist and professional medical fact-checker specializing in diabetes care. Your core responsibility is to verify the accuracy of diabetes-related information from social media short videos, extract standardized medical triples (Subject - Predicate - Object), and ensure consistency with authoritative medical knowledge graph. 
    Requirements:
    1. Format: must be a standard json list of objects: [("S": "Subject", "P": "Predicate", "O": "Object"}].
    2. Predicate (P) should be professional and standardized, such as: treats, causes, symptom of, prevents, belongs to, contraindicated for, increases risk of, lowers blood glucose, associated with, and so on
    3. Language: The output must be in English.
    4. If no clear triplets are found, return an empty list [].

    For the input social media diabetes video content, strictly follow the steps below to ensure logical rigor:
    1. Claim Parsing: Extract the core medical claim (filter out irrelevant information such as emotional expressions, advertising slogans, or personal stories); if abbreviations/acronyms exist (e.g., T1D, T2D), first convert them to standard medical terms.
    2. Triple Extraction: Generate standardized triples in the format "Subject - Predicate - Object", where:
    Subject: Diabetes subtype (e.g., Type 1 diabetes), medication (e.g., Metformin), food (e.g., oats), intervention measure (e.g., carbohydrate intake control), etc.
    Predicate: Treatment Requirement, Dietary Guidance, Recommended Dosage, Effect, Pathophysiological Cause, Monitoring Requirement, etc.
    Object: Specific operation, dose, effect, or cause (e.g., Daily insulin injection, 500mg twice daily, Blood glucose control), etc.

    Below are 5 standard examples of diabetes-related triple extraction and accuracy verification. Learn the format of standardized triples, reasoning logic, and accuracy judgment criteria:
    1. Video Content: “Hello, my name is Eva Bess. Type 1 diabetes patients need to inject insulin every day because their pancreas can’t produce insulin on its own.”
    Step 1: Parse core claim: Type 1 diabetes requires daily insulin injection because of pancreatic insulin deficiency. (“Hello, my name is Eva Bess” is not related to diabetes)
    Step 2: Extract standardized triple: “Type 1 diabetes - Treatment Requirement - Daily insulin injection”,  “Type 1 diabetes - Pathophysiological Cause - Pancreatic insulin secretion deficiency”
    2. Video Content: “Eating oats helps control blood sugar in Type 2 diabetes.”
    Step 1: Parse core claim: Type 2 diabetes patients eating oats aids blood glucose control
    Step 2: Extract standardized triple: “Type 2 diabetes - Dietary Guidance - Intake of oats”,  “Oats - Effect - Blood glucose control”
    3. Video Content: “T1D patients don’t need to monitor blood sugar regularly.”
    Step 1: Parse core claim: Type 1 diabetes patients do not require regular blood glucose monitoring  (Note: T1D = Type 1 diabetes abbreviation)
    Step 2: Extract standardized triple: “Type 1 diabetes - Monitoring Requirement - No regular blood glucose monitoring”
    4. Video Content: “My name is Megan. I've been in a monogamist relationship with Minimadge since 1997. I've been able to fold in my diabetes management into my bigger routine. I am juggling things all the time. The technology is so seamless with my life. I have one less ball in the air. I've stayed with Mini Med all these years because it feels like a family to me. I choose Mini Med because it's always got my back. I can't imagine my life without them.”
    Step 1: Parse core claim: Long-term integration of advanced diabetes technology (such as the MiniMed insulin pump system) effectively reduces the daily burden of diabetes management and improves the patient's quality of life. (Other information is not related to diabetes)
    Step 2: Extract standardized triple: “Diabetes Patient - Uses - MiniMed Diabetes Management Technology”, “Insulin Pump System - Integrates - Daily Diabetes Management Routine”, “Automated Medical Technology - Reduces - Diabetes Treatment Burden”, “Long-term Technology Use - Improves - Patient Treatment Adherence”
    5. Video Content: “Attention, diabetic patients! The latest SGLT-2 inhibitors (such as Dapagliflozin) not only lower blood sugar but also protect the heart and kidneys, making them a first-line medication recommended by current guidelines. However, every medicine has side effects. If you start taking this drug, you must completely stop eating all carbohydrates and switch to a vegan diet; this will permanently cure your Type 2 diabetes and allow you to stop taking medication forever.”
    Step 1: Parse core claim: SGLT-2 inhibitors provide dual benefits of glycemic control and cardio-renal protection; SGLT-2 inhibitor therapy requires a zero-carb and vegan-only diet; This combination is a method to permanently cure Type 2 diabetes, allowing for the discontinuation of all medications. (Other information is not related to diabetes)
    Step 2: Extract standardized triple: “SGLT-2 Inhibitors - Provide - Cardio-renal protection”,  “SGLT-2 Inhibitors - Require - Zero-carb diet”, “Type 2 Diabetes - Through medication and diet - Permanent cure”, “Dapagliflozin - Belongs to - SGLT-2 Inhibitors”
    """

    # Few-shot Example in English
    user_example = "Text: 'Metformin can effectively lower blood glucose levels in patients with Type 2 Diabetes.'\nOutput:"
    assistant_example = '[{"S": "Metformin", "P": "lowers_blood_glucose", "O": "Type 2 Diabetes"}]'

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_example},
                {"role": "assistant", "content": assistant_example},
                {"role": "user", "content": f"Text to process: '{chunk_text}'\nOutput:"}
            ],
            response_format={ "type": "json_object" }
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error during API call: {e}")
        return "[]"

In [9]:
df = pd.read_csv("dataset/alltext_cleaned.csv", encoding="utf-8")

# 检查 'text' 列是否存在
if "text" not in df.columns:
    raise ValueError("The column named 'text' was not found in the CSV file. Please check the column name.")

print(f"A total of {len(df)} pieces of text were read.")

A total of 330 pieces of text were read.


In [ ]:
triplets_list = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting triplets"):
    text = row["text"]
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        triplets_list.append("[]")  # 空文本返回空列表
        continue
    
    result_str = get_triplets_from_ai(text)
    # 尝试解析一下，确保返回的是合法 JSON（可选）
    try:
        # 验证是否能解析为 JSON，如果失败则存储空列表
        json.loads(result_str)
        triplets_list.append(result_str)
    except json.JSONDecodeError:
        print(f"Warning: The {idx}th item returned is not a valid JSON and has been emptied. Content: {result_str[:100]}")
        triplets_list.append("[]")
    
    time.sleep(0.1)  # 控制请求频率

df["triplets_json"] = triplets_list

Extracting triplets: 100%|██████████| 330/330 [45:55<00:00,  8.35s/it]


In [ ]:
output_file = "output_data/text_with_triplets_ver3.json"
df.to_json(output_file, orient="records", force_ascii=False, indent=2)
print(f"The result has been saved to {output_file}")

The result has been saved to output_data/text_with_triplets.json


In [ ]:
expanded_rows = []
for idx, row in df.iterrows():
    try:
        triplets = json.loads(row["triplets_json"])
        for t in triplets:
            expanded_rows.append({
                "source_row": idx,
                "subject": t.get("S", ""),
                "predicate": t.get("P", ""),
                "object": t.get("O", "")
            })
    except:
        # 如果某行 JSON 解析失败，跳过
        continue

expanded_df = pd.DataFrame(expanded_rows)
# expanded_df.to_csv("expanded_triplets.csv", index=False, encoding="utf-8")
expanded_df.to_json(
    "output_data/expanded_triplets_ver3.json",
    orient="records",
    force_ascii=False,
    indent=2
)
print("The expanded triples have been saved.")

The expanded triples have been saved.


## Clean Triples

In [ ]:
import pandas as pd
import re

# 输入输出路径
input_path = "output_data/expanded_triplets_ver3.json"
output_path = "output_data/expanded_triplets_ver3_clean.json"

# 读取数据
df = pd.read_json(input_path)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 小写
    text = text.lower()
    
    # 下划线 → 空格
    text = text.replace("_", " ")
    
    # 去多余空格
    text = re.sub(r"\s+", " ", text)
    
    # 去首尾标点
    text = text.strip(".,;:!? ")
    
    return text

# 应用清洗（不改变结构）
df["subject"] = df["subject"].apply(clean_text)
df["predicate"] = df["predicate"].apply(clean_text)
df["object"] = df["object"].apply(clean_text)

# 输出
df.to_json(
    output_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print("Cleaned triples saved (no deduplication).")
print("Total rows:", len(df))
print(df.head())